In [ ]:
import numpy as np
from matplotlib import pyplot as plt
import pandas as pd
from sklearn.metrics import confusion_matrix
import pynapple as nap
from spatial_manifolds.toroidal import *
from spatial_manifolds.behaviour_plots import *
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.ticker import MaxNLocator
from spatial_manifolds.mlencoding import *
from spatial_manifolds.circular_decoder import circular_decoder, cross_validate_decoder, cross_validate_decoder_time, circular_nanmean
from spatial_manifolds.data.curation import curate_clusters
from scipy.stats import zscore
from spatial_manifolds.util import gaussian_filter_nan
from spatial_manifolds.predictive_grid import compute_travel_projected, wrap_list
from spatial_manifolds.behaviour_plots import *
from spatial_manifolds.detect_grids import *
from spatial_manifolds.brainrender_helper import *

import warnings
warnings.filterwarnings('ignore')
%load_ext autoreload
%autoreload 2
%matplotlib inline

In [ ]:
fig_path = '/Users/harryclark/Documents/figs/FIGURE1/'
mouse = 29
day = 23

# good examples include 
#mice = [25, 25, 26, 27, 29, 28]
#days = [25, 24, 18, 26, 23, 25]

In [ ]:
# Plot theta traces for multiple 2 s windows across channels (every 10th channel)
channels = [f"CH{i}" for i in range(1, 385, 10)]  # use every 10th channel for clarity

resample_bs = 10  # ms, for smoother traces (optional)
time_bs_lfp = 50  # ms
window_length_sec = 2.0
bins_per_sec = int(1000 / resample_bs if resample_bs is not None else time_bs_lfp)
window_bins = int(window_length_sec * bins_per_sec)

# Define a few different 2 s snippets (in bins from the start of the recording)
snippet_start_bins = [0, 10*window_bins, 50*window_bins]  # 0–2 s, 2–4 s, 4–6 s (approx.)

for snippet_idx, window_start_idx in enumerate(snippet_start_bins):
    window_end_idx = window_start_idx + window_bins

    theta_traces = []
    for ch in channels:
        print(f"Computing theta trace for {ch}, snippet {snippet_idx + 1}...")
        trace = get_theta_trace(
            mouse=29,
            day=23,
            cluster_id=1,
            time_bs=time_bs_lfp,
            resample_bs=resample_bs,
            vr_type='VR',
            channel=ch
        )
        theta_traces.append(np.array(trace)[window_start_idx:window_end_idx])
    print(f"Length of trace for snippet {snippet_idx + 1}: {len(np.array(trace))}")
    
    theta_traces = np.array(theta_traces)  # shape: (n_channels, window_bins)
    n_channels, n_time = theta_traces.shape  # n_time should equal window_bins

    plt.figure(figsize=(12, 6))
    offset = np.nanmax(np.abs(theta_traces)) * 1.2 if np.isfinite(theta_traces).any() else 1.0

    t_idx = np.arange(n_time) * (resample_bs / 1000 if resample_bs is not None else time_bs_lfp / 1000)

    for i, ch in enumerate(channels):
        plt.plot(t_idx, theta_traces[i] + i * offset, lw=0.5, label=ch if i < 10 else None)

    # Simple y-ticks: label every plotted channel
    yticks = [i * offset for i in range(n_channels)]
    yticklabels = [ch for ch in channels]
    plt.yticks(yticks, yticklabels)

    start_time_sec = window_start_idx / bins_per_sec
    plt.xlabel("Time within window (s)")
    plt.title(
        f"Theta traces for channels (2 s window starting at {start_time_sec:.1f} s, "
        f"binned at {time_bs_lfp} ms, resampled at {resample_bs} ms)"
    )
    plt.tight_layout()
    plt.show()

In [ ]:
gcs, ngs, ns, sc, ngs_ns, all = cell_classification_of1(mouse, day, percentile_threshold=95) # subset
rc, rsc, vr_ns = cell_classification_vr(mouse, day)
mec, para, pre, sub, vis, cere, other, all_by_anatomy = cell_classification_anatomy(mouse, day)
g_m_ids, g_m_cluster_ids = HDBSCAN_grid_modules(gcs, all, mouse, day, min_cluster_size=3, cluster_selection_epsilon=3, 
                                                figpath=fig_path, curate_with_vr=False, curate_with_brain_region=True, plot_curate=False) # create grid modules using HDBSCAN    

plot_grid_modules_rate_maps(gcs, g_m_ids, g_m_cluster_ids, mouse, day, figpath=fig_path)

# we now have cluster ids classified into modules, non grid spatial cells and non spatial cells 
# as defined by activity in the open field
g_m_cluster_ids = sorted(g_m_cluster_ids, key=len, reverse=True) 
cluster_ids_by_group = []
cluster_ids_by_group.extend(g_m_cluster_ids) # grid cells by module [0,1,2...]
cluster_ids_by_group.append(ngs.cluster_id.values.tolist()) # non grid spatial [-4]
cluster_ids_by_group.append(ns.cluster_id.values.tolist()) # non spatial cells [-3]
cluster_ids_by_group.append(gcs.cluster_id.values.tolist()) # all grid cells [-2]
cluster_ids_by_group.append(sc.cluster_id.values.tolist()) # speed cells [-1]

for m, cluster_ids in enumerate(cluster_ids_by_group):
    plot_vr_rate_maps(mouse, day, cluster_ids, label=f'{m}', figpath=fig_path)

plot_vr_rate_maps(mouse, day, np.setdiff1d(gcs.cluster_id.values, g_m_cluster_ids[0]), label=f'nocomodular_grid_cells', figpath=fig_path)

In [ ]:
tcs, tcs_time, _, last_ephys_bin, beh, clusters = compute_vr_tcs(mouse,day, apply_zscore=False, apply_guassian_filter=False)
tcs_of, tcs_time_of, beh_of, clusters_of, ep_OF = compute_of_tcs(mouse,day, apply_zscore=False, apply_guassian_filter=False, optimal_shift=gcs.travel[0])
last_ephys_time_bin = clusters[clusters.index[0]].count(bin_size=time_bs, time_units = 'ms').index[-1]

# time binned variables for later
ep = nap.IntervalSet(start=0, end=last_ephys_time_bin, time_units = 's')
speed_in_time = np.array(beh['S'].bin_average(bin_size=time_bs, time_units = 'ms', ep=ep))
dt_in_time = np.array(beh['travel'].bin_average(bin_size=time_bs, time_units = 'ms', ep=ep)-((beh['trial_number'][0]-1)*tl))
pos_in_time = dt_in_time%tl
trial_number_in_time = (dt_in_time//tl)+beh['trial_number'][0]
print(len(pos_in_time), len(speed_in_time), len(trial_number_in_time))

x_pos_in_time_OF = np.array(beh_of['head_x'].bin_average(bin_size=time_bs, time_units = 'ms', ep=ep_OF))
y_pos_in_time_OF = np.array(beh_of['head_y'].bin_average(bin_size=time_bs, time_units = 'ms', ep=ep_OF))

if np.any(np.isnan(pos_in_time)):
    series = pd.Series(dt_in_time)
    filled_series = series.ffill().bfill()
    dt_in_time = np.array(filled_series)
    pos_in_time = dt_in_time%tl
    trial_number_in_time = (dt_in_time//tl)+beh['trial_number'][0]

if np.any(np.isnan(speed_in_time)):
    series = pd.Series(speed_in_time)
    filled_series = series.ffill().bfill()
    speed_in_time = np.array(speed_in_time)

if np.any(np.isnan(x_pos_in_time_OF)):
    series = pd.Series(x_pos_in_time_OF)
    filled_series = series.ffill().bfill()
    x_pos_in_time_OF = np.array(filled_series)

if np.any(np.isnan(y_pos_in_time_OF)):
    series = pd.Series(y_pos_in_time_OF)
    filled_series = series.ffill().bfill()
    y_pos_in_time_OF = np.array(filled_series)

In [ ]:
xgb_history = MLencoding(tunemodel = 'xgboost',
                         cov_history = True, spike_history=False, # We can choose!
                         window = time_bs, #this dataset has 100ms time bins
                         n_filters = 5,
                         max_time = 1000)

In [ ]:
test_grid_cell_1 = 214  # cm
test_grid_cell_2 = 332  # ncm
test_ngs_cell_1 = 340   # ngs

import random

# Prepare grid predictors (remove test cells if present, then shuffle and select 10)
grid_predictors = g_m_cluster_ids[0].copy()
for test_id in [test_grid_cell_1, test_grid_cell_2, test_ngs_cell_1]:
    if test_id in grid_predictors:
        grid_predictors.remove(test_id)
random.shuffle(grid_predictors)
grid_predictors = grid_predictors[:10]

# Prepare NGS predictors (remove test cells if present, then shuffle and select 10)
ngs_predictors = ngs.cluster_id.values.tolist()
for test_id in [test_grid_cell_1, test_grid_cell_2, test_ngs_cell_1]:
    if test_id in ngs_predictors:
        ngs_predictors.remove(test_id)
random.shuffle(ngs_predictors)
ngs_predictors = ngs_predictors[:10]

In [ ]:
def predict_xgboost_traces(
    target_id,
    tcs_time,
    pos_in_time,
    speed_in_time,
    grid_predictors,
    ngs_predictors,
    xgb_history,
    gaussian_filter_nan,
    n_grid=10,
    n_ngs=10,
    mouse=29,
    day=23
):
    """
    Predicts activity for a target cell using XGBoost with various covariate sets.

    Returns a dictionary of predictions and cross-validated pR2 values.
    """
    y = np.array(tcs_time[target_id])
    y_smoothed = gaussian_filter_nan(y, sigma=3)
    
    # --- LFP (theta) ---
    lfp = get_theta_trace(
        mouse=mouse,
        day=day,
        cluster_id=target_id,
        time_bs=50,
        resample_bs=time_bs,  # assumes global time_bs
        vr_type='VR'
    )
    # Make lfp the same length as y by trimming or zero-padding as needed
    if len(lfp) < len(y):
        lfp = np.pad(lfp, (0, len(y) - len(lfp)), mode='constant')
    else:
        lfp = lfp[:len(y)]

    # --- SPEED ---
    speed = np.array(speed_in_time)
    # Make speed the same length as y by trimming or zero-padding as needed
    if len(speed) < len(y):
        speed = np.pad(speed, (0, len(y) - len(speed)), mode='constant')
    else:
        speed = speed[:len(y)]

    # --- POSITION ONLY ---
    pos = np.array(pos_in_time)
    if len(pos) < len(y):
        pos = np.pad(pos, (0, len(y) - len(pos)), mode='constant')
    else:
        pos = pos[:len(y)]

    # --- GRID CELLS ---
    Xg_1 = np.stack([np.array(tcs_time[grid_predictors[0]])]).T
    Xpos_g_1 = np.stack([pos, np.array(tcs_time[grid_predictors[0]])]).T

    Xg_5 = np.column_stack([np.array(tcs_time[cid]) for cid in grid_predictors[:5]])
    Xpos_g_5 = np.column_stack((pos, Xg_5))

    Xg_10 = np.column_stack([np.array(tcs_time[cid]) for cid in grid_predictors[:n_grid]])
    Xpos_g_10 = np.column_stack((pos, Xg_10))

    # --- NGS CELLS ---
    Xngs_1 = np.stack([np.array(tcs_time[ngs_predictors[0]])]).T
    Xpos_ngs_1 = np.stack([pos, np.array(tcs_time[ngs_predictors[0]])]).T

    Xngs_5 = np.column_stack([np.array(tcs_time[cid]) for cid in ngs_predictors[:5]])
    Xpos_ngs_5 = np.column_stack((pos, Xngs_5))

    Xngs_10 = np.column_stack([np.array(tcs_time[cid]) for cid in ngs_predictors[:n_ngs]])
    Xpos_ngs_10 = np.column_stack((pos, Xngs_10))

    # --- POSITION / LFP / SPEED BASES ---
    Xpos = pos[:, None]
    Xlfp = lfp[:, None]
    Xspeed = speed[:, None]

    Xpos_lfp = np.column_stack((pos, lfp))
    Xpos_speed = np.column_stack((pos, speed))
    Xspeed_lfp = np.column_stack((speed, lfp))
    Xpos_speed_lfp = np.column_stack((pos, speed, lfp))

    # --- LFP WITH CELLS ---
    Xg_10_lfp = np.column_stack((Xg_10, lfp))
    Xngs_10_lfp = np.column_stack((Xngs_10, lfp))
    Xpos_g_10_lfp = np.column_stack((pos, Xg_10, lfp))
    Xpos_ngs_10_lfp = np.column_stack((pos, Xngs_10, lfp))

    # --- SPEED WITH CELLS (± position / LFP) ---
    # Grid
    Xg_10_speed = np.column_stack((Xg_10, speed))
    Xg_10_speed_lfp = np.column_stack((Xg_10, speed, lfp))
    Xpos_g_10_speed = np.column_stack((pos, Xg_10, speed))
    Xpos_g_10_speed_lfp = np.column_stack((pos, Xg_10, speed, lfp))

    # NGS
    Xngs_10_speed = np.column_stack((Xngs_10, speed))
    Xngs_10_speed_lfp = np.column_stack((Xngs_10, speed, lfp))
    Xpos_ngs_10_speed = np.column_stack((pos, Xngs_10, speed))
    Xpos_ngs_10_speed_lfp = np.column_stack((pos, Xngs_10, speed, lfp))

    # --- FIT MODELS ---
    results = {}

    # Position only
    results['pos'] = xgb_history.fit_cv(Xpos, y, verbose=0, continuous_folds=True)

    # LFP only
    results['lfp'] = xgb_history.fit_cv(Xlfp, y, verbose=0, continuous_folds=True)

    # Speed only
    results['speed'] = xgb_history.fit_cv(Xspeed, y, verbose=0, continuous_folds=True)

    # Position + LFP / speed / both
    results['pos_lfp'] = xgb_history.fit_cv(Xpos_lfp, y, verbose=0, continuous_folds=True)
    results['pos_speed'] = xgb_history.fit_cv(Xpos_speed, y, verbose=0, continuous_folds=True)
    results['speed_lfp'] = xgb_history.fit_cv(Xspeed_lfp, y, verbose=0, continuous_folds=True)
    results['pos_speed_lfp'] = xgb_history.fit_cv(Xpos_speed_lfp, y, verbose=0, continuous_folds=True)

    # Grid predictors
    results['g1'] = xgb_history.fit_cv(Xg_1, y, verbose=0, continuous_folds=True)
    results['g5'] = xgb_history.fit_cv(Xg_5, y, verbose=0, continuous_folds=True)
    results['g10'] = xgb_history.fit_cv(Xg_10, y, verbose=0, continuous_folds=True)
    results['g10_lfp'] = xgb_history.fit_cv(Xg_10_lfp, y, verbose=0, continuous_folds=True)

    # Grid predictors + position
    results['pos_g1'] = xgb_history.fit_cv(Xpos_g_1, y, verbose=0, continuous_folds=True)
    results['pos_g5'] = xgb_history.fit_cv(Xpos_g_5, y, verbose=0, continuous_folds=True)
    results['pos_g10'] = xgb_history.fit_cv(Xpos_g_10, y, verbose=0, continuous_folds=True)
    results['pos_g10_lfp'] = xgb_history.fit_cv(Xpos_g_10_lfp, y, verbose=0, continuous_folds=True)

    # Grid predictors + speed / speed+LFP / pos+speed / pos+speed+LFP
    results['g10_speed'] = xgb_history.fit_cv(Xg_10_speed, y, verbose=0, continuous_folds=True)
    results['g10_speed_lfp'] = xgb_history.fit_cv(Xg_10_speed_lfp, y, verbose=0, continuous_folds=True)
    results['pos_g10_speed'] = xgb_history.fit_cv(Xpos_g_10_speed, y, verbose=0, continuous_folds=True)
    results['pos_g10_speed_lfp'] = xgb_history.fit_cv(Xpos_g_10_speed_lfp, y, verbose=0, continuous_folds=True)

    # NGS predictors
    results['ngs1'] = xgb_history.fit_cv(Xngs_1, y, verbose=0, continuous_folds=True)
    results['ngs5'] = xgb_history.fit_cv(Xngs_5, y, verbose=0, continuous_folds=True)
    results['ngs10'] = xgb_history.fit_cv(Xngs_10, y, verbose=0, continuous_folds=True)
    results['ngs10_lfp'] = xgb_history.fit_cv(Xngs_10_lfp, y, verbose=0, continuous_folds=True)

    # NGS predictors + position
    results['pos_ngs1'] = xgb_history.fit_cv(Xpos_ngs_1, y, verbose=0, continuous_folds=True)
    results['pos_ngs5'] = xgb_history.fit_cv(Xpos_ngs_5, y, verbose=0, continuous_folds=True)
    results['pos_ngs10'] = xgb_history.fit_cv(Xpos_ngs_10, y, verbose=0, continuous_folds=True)
    results['pos_ngs10_lfp'] = xgb_history.fit_cv(Xpos_ngs_10_lfp, y, verbose=0, continuous_folds=True)

    # NGS predictors + speed / speed+LFP / pos+speed / pos+speed+LFP
    results['ngs10_speed'] = xgb_history.fit_cv(Xngs_10_speed, y, verbose=0, continuous_folds=True)
    results['ngs10_speed_lfp'] = xgb_history.fit_cv(Xngs_10_speed_lfp, y, verbose=0, continuous_folds=True)
    results['pos_ngs10_speed'] = xgb_history.fit_cv(Xpos_ngs_10_speed, y, verbose=0, continuous_folds=True)
    results['pos_ngs10_speed_lfp'] = xgb_history.fit_cv(Xpos_ngs_10_speed_lfp, y, verbose=0, continuous_folds=True)

    # Also return the true and smoothed target for plotting
    results['y'] = y
    results['y_smoothed'] = y_smoothed

    results[' '] = (y*np.nan, np.zeros(10).tolist()) # Placeholder for any additional info if needed
    return results

In [ ]:
# Example for three target cells
target_cell_names = [f'Unit {test_grid_cell_1}', f'Unit {test_grid_cell_2}', f'Unit {test_ngs_cell_1}']
target_cell_ids = [test_grid_cell_1, test_grid_cell_2, test_ngs_cell_1]

# Run predictions for each target cell and collect results
results_dict = {}
for cell_name, cell_id in zip(target_cell_names, target_cell_ids):
    results_dict[cell_name] = predict_xgboost_traces(
        target_id=cell_id,
        tcs_time=tcs_time,
        pos_in_time=pos_in_time,
        speed_in_time=speed_in_time,
        grid_predictors=grid_predictors,
        ngs_predictors=ngs_predictors,
        xgb_history=xgb_history,
        gaussian_filter_nan=gaussian_filter_nan,
        n_grid=10,
        n_ngs=10
    )

In [ ]:
def make_tidy_labels(covariate_labels):
    """
    Map raw covariate labels to human-readable tidy labels.

    Any covariate not in the mapping is returned unchanged.
    """
    mapping = {
        # Base covariates
        'pos': 'Pos (P)',
        'lfp': 'LFP',
        'speed': 'Speed (S)',
        ' ': ' ',  # for the empty separator

        # Base combinations
        'pos_lfp': 'LFP+P',
        'pos_speed': 'S+P',
        'speed_lfp': 'S+LFP',
        'pos_speed_lfp': 'S+LFP+P',

        # Grid-only
        'g1': 'GCn=1',
        'g5': 'GCn=5',
        'g10': 'GCn=10',

        # Grid + LFP
        'g10_lfp': 'LFP+GCn=10',

        # Grid + speed
        'g10_speed': 'S+GCn=10',
        'g10_speed_lfp': 'S+LFP+GCn=10',

        # Grid + position (± LFP / speed)
        'pos_g1': 'P+GCn=1',
        'pos_g5': 'P+GCn=5',
        'pos_g10': 'P+GCn=10',
        'pos_g10_lfp': 'LFP+P+GCn=10',
        'pos_g10_speed': 'S+P+GCn=10',
        'pos_g10_speed_lfp': 'S+LFP+P+GCn=10',

        # NGS-only
        'ngs1': 'NGSn=1',
        'ngs5': 'NGSn=5',
        'ngs10': 'NGSn=10',

        # NGS + LFP
        'ngs10_lfp': 'LFP+NGSn=10',

        # NGS + speed
        'ngs10_speed': 'S+NGSn=10',
        'ngs10_speed_lfp': 'S+LFP+NGSn=10',

        # NGS + position (± LFP / speed)
        'pos_ngs1': 'P+NGSn=1',
        'pos_ngs5': 'P+NGSn=5',
        'pos_ngs10': 'P+NGSn=10',
        'pos_ngs10_lfp': 'LFP+P+NGSn=10',
        'pos_ngs10_speed': 'S+P+NGSn=10',
        'pos_ngs10_speed_lfp': 'S+LFP+P+NGSn=10',
    }
    tidy = []
    for cov in covariate_labels:
        tidy.append(mapping.get(cov, cov))
    return tidy


def plot_xgboost_traces(
    results_dict,
    target_cell_names,
    covariate_labels=[
        'pos', 'speed', 'lfp', 
        'pos_speed', 'pos_lfp', 'speed_lfp', 'pos_speed_lfp',
        ' ',
        'g1', 'g5', 'g10', 'g10_lfp',
        'pos_g1', 'pos_g5', 'pos_g10', 'pos_g10_lfp',
        'g10_speed', 'g10_speed_lfp', 'pos_g10_speed', 'pos_g10_speed_lfp',
        ' ',
        'ngs1', 'ngs5', 'ngs10', 'ngs10_lfp',
        'pos_ngs1', 'pos_ngs5', 'pos_ngs10', 'pos_ngs10_lfp',
        'ngs10_speed', 'ngs10_speed_lfp', 'pos_ngs10_speed', 'pos_ngs10_speed_lfp',
    ],
    tidy_covariate_labels=None,
    true_labels=['y', 'y_smoothed'],
    window=None,
    figsize=(15, 4),
    gain=[1, 1, 1],
    text_offset=0.0,
    offset_jump=0.0,
    savepath=None,
    time_bs=100,  # ms per bin, adjust as needed
):
    """
    Plot true and predicted traces for target cells side by side using results_dict directly.

    Each covariate prediction is offset vertically for clarity.
    true_labels are plotted at the top, overlayed, in black and grey (same offset).
    Covariate label on left, mean pR2 on right. No axes.

    Offsets are fixed and identical across all subplots.
    Colors: test_grid_cell_1 and test_grid_cell_2 = red, test_ngs_cell_1 = blue.
    text_offset: vertical shift of text annotations (in units of offset_step).
    offset_jump: optional extra spacing between the true trace and the first covariate.
    tidy_covariate_labels: if None, they are generated from covariate_labels via make_tidy_labels.
    """
    import matplotlib.pyplot as plt
    import numpy as np

    def subscript_n(label):
        import re

        # Replace "n=10" → "$_{n=10}$" inside the label, e.g. "GCn=10" → "GC$_{n=10}$"
        def repl(match):
            n_val = match.group(1)
            return r'$_{n=' + n_val + '}$'

        return re.sub(r'n=([0-9]+)', repl, label)

    # Build tidy labels if not provided
    if tidy_covariate_labels is None:
        tidy_covariate_labels = make_tidy_labels(covariate_labels)
    if len(tidy_covariate_labels) != len(covariate_labels):
        raise ValueError(
            f"tidy_covariate_labels must have the same length as covariate_labels "
            f"({len(tidy_covariate_labels)} vs {len(covariate_labels)})"
        )

    # Ordered labels: first the main true trace label, then all covariates in the user-given order
    ordered_labels = [true_labels[0]] + list(covariate_labels)
    n_offsets = len(ordered_labels)

    # Compute a global offset step based on all true traces
    all_true_traces = [results_dict[cell]["y"] for cell in target_cell_names]
    if len(all_true_traces) > 0:
        print(f"Length of y (time bins): {len(all_true_traces[0])}")
    if window is not None:
        all_true_traces = [trace[window[0]:window[1]] for trace in all_true_traces]
    global_max = np.nanmax([np.nanmax(trace) for trace in all_true_traces])
    offset_step = global_max * 1.2 if global_max > 0 else 1

    # Build vertical offsets; allow a jump after the first covariate if desired
    offsets = []
    for i, label in enumerate(ordered_labels):
        jump = 0.0
        if i == 1:  # right after the true trace
            jump = offset_jump * offset_step
        if i == 0:
            offsets.append(offset_step * (n_offsets - 1 - i))
        else:
            offsets.append(offsets[-1] - offset_step - jump)

    # Assign colors by test cell: red for grid cells, blue for ngs
    color_map_by_cell = {}
    for cell_name in target_cell_names:
        if '340' in str(cell_name).lower():
            color_map_by_cell[cell_name] = '#3171ae'
        else:
            color_map_by_cell[cell_name] = '#c04744'
    true_label_colors = ['grey', 'black']

    n_cells = len(target_cell_names)
    fig, axes = plt.subplots(1, n_cells, figsize=figsize, sharey=True)
    if n_cells == 1:
        axes = [axes]

    for i, cell_name in enumerate(target_cell_names):
        ax = axes[i]
        true_trace = results_dict[cell_name]["y"]
        if window is not None:
            t_idx = np.arange(window[0], window[1])
            true_trace = true_trace[window[0]:window[1]]
        else:
            t_idx = np.arange(len(true_trace))

        # Plot true labels at the top, overlayed, with same offset
        true_offset = offsets[0]
        for j, label in enumerate(true_labels):
            if label in results_dict[cell_name]:
                trace = results_dict[cell_name][label]
                if window is not None:
                    trace = trace[window[0]:window[1]]
                ax.plot(
                    t_idx,
                    trace * 2 + true_offset,
                    lw=2,
                    color=true_label_colors[j],
                    alpha=0.8,
                )
                if i == 0:
                    ax.text(
                        t_idx[0] - (0.08 * len(t_idx)),  # shift left of y axis label
                        np.nanmean(trace * 2 + true_offset) + text_offset * offset_step,
                        label,
                        color=true_label_colors[j],
                        va='center',
                        ha='right',
                        fontsize=15,
                    )

        # Plot predictions for each covariate, using the same order as covariate_labels
        for j, cov_label in enumerate(covariate_labels):
            idx = ordered_labels.index(cov_label)
            pred_trace = results_dict[cell_name][cov_label][0]
            if window is not None:
                pred_trace = pred_trace[window[0]:window[1]]
            pr2 = np.nanmean(results_dict[cell_name][cov_label][1])
            color = color_map_by_cell[cell_name]
            offset = offsets[idx]

            tidy_label = subscript_n(tidy_covariate_labels[j])

            if i == 0:
                ax.text(
                    t_idx[0] - (0.08 * len(t_idx)),  # shift left of y axis label
                    np.nanmean(pred_trace + offset) + text_offset * offset_step,
                    tidy_label,
                    color=color,
                    va='center',
                    ha='right',
                    fontsize=15,
                )
            print(f"Plotting {cov_label} for {cell_name} with pR2={pr2:.4f}")
            print(f'shape of pred_trace: {pred_trace.shape}, length of t_idx: {len(t_idx)}')
            ax.plot(t_idx, pred_trace * gain[i] + offset, lw=1.5, alpha=0.8, color=color)
            ax.text(
                t_idx[-1] + (0.01 * len(t_idx)),
                np.nanmean(pred_trace + offset) + text_offset * offset_step,
                f'{pr2:.2f}',
                color=color,
                va='center',
                ha='left',
                fontsize=10,
            )

        ax.set_title(str(cell_name), fontsize=16)
        ax.set_xticks([])
        ax.set_yticks([])
        ax.axis('off')

    # Add y axis label to the leftmost plot
    axes[0].set_ylabel('Target prediction', fontsize=18, labelpad=60)

    # Plot a 5 s scale bar on the leftmost axis
    if window is not None:
        t_start = window[0]
    else:
        t_start = 0
    t_per_bin = time_bs / 1000.0  # convert ms to s
    bar_len = int(5 / t_per_bin)
    axes[0].plot([t_start, t_start + bar_len], [offsets[-1], offsets[-1]], color='black', lw=2)

    plt.tight_layout()
    if savepath is not None:
        plt.savefig(savepath)
    plt.show()

In [ ]:
results_dict['Unit 214']['pos'][0]

In [ ]:
b=-30000
# Now you can call the plotting function
plot_xgboost_traces(results_dict, 
                    target_cell_names,
                    window=(24150, 26150), 
                    figsize=(10, 8),
                    gain=[5, 12, 6],
                    text_offset=0.25,
                    offset_jump=0.3,
                    savepath=f'/Users/harryclark/Documents/spatial-manifolds/scripts/figures/figure_3_xgboost/M{mouse}D{day}_xgboost_traces_with_theta.pdf')

In [ ]:
results_dict['Unit 214'][' ']

In [ ]:
for b in [-54000, -50000, -46000, -42000, -38000, -34000, -30000, -26000, -22000, -18000, -14000, -10000, -6000, -2000]:
    print(b)
    
    # Now you can call the plotting function
    plot_xgboost_traces(results_dict, 
                        target_cell_names,
                        window=(54200+b, 57200+b), 
                        figsize=(15, 9),
                        gain=[5, 12, 6],
                        text_offset=0.25,
                        offset_jump=0.3,
                        savepath=fig_path + f'M{mouse}D{day}_xgboost_traces_with_theta.pdf')

I need to work out how to add theta as a covariate
